# 03 — Burnout Risk Score
Flag employees combining `OverTime = Yes` with a low `WorkLifeBalance` (≤ 2), and assign a Low / Medium / High burnout risk level.

In [1]:
import pandas as pd
df = pd.read_csv('../data/processed/engagement_scored.csv')
df[['OverTime', 'WorkLifeBalance']].describe(include='all')

,OverTime,WorkLifeBalance
count,1470,1470.000000
unique,2,NaN
top,No,NaN
freq,1054,NaN
mean,NaN,2.761224
std,NaN,0.706476
min,NaN,1.000000
25%,NaN,2.000000
50%,NaN,3.000000
75%,NaN,3.000000


## Core burnout flag
The two ingredients specified in the project brief: overtime work and poor work-life balance.

In [2]:
df['OverTimeFlag'] = (df['OverTime'] == 'Yes').astype(int)
df['LowWLBFlag'] = (df['WorkLifeBalance'] <= 2).astype(int)
df['BurnoutFlag'] = ((df['OverTimeFlag'] == 1) & (df['LowWLBFlag'] == 1)).astype(int)
df['BurnoutFlag'].value_counts()

BurnoutFlag
0    1344
1     126
Name: count, dtype: int64

## Burnout Risk levels
Beyond the binary flag, build a small risk score using overtime, work-life balance, and low engagement together, so 'at risk but not yet flagged' employees (e.g. overtime + medium balance + low engagement) also surface.

In [3]:
def burnout_risk(row):
    score = 0
    if row['OverTime'] == 'Yes':
        score += 1
    if row['WorkLifeBalance'] <= 2:
        score += 2
    elif row['WorkLifeBalance'] == 3:
        score += 1
    if row['EngagementIndex'] < 0.34:
        score += 1
    if score >= 3:
        return 'High'
    elif score >= 1:
        return 'Medium'
    else:
        return 'Low'

df['BurnoutRisk'] = df.apply(burnout_risk, axis=1)
df['BurnoutRisk'].value_counts()

BurnoutRisk
Medium    1178
High       191
Low        101
Name: count, dtype: int64

## Cross-check: burnout risk vs engagement tier

In [4]:
pd.crosstab(df['BurnoutRisk'], df['EngagementTier'])

EngagementTier,High,Low,Medium
BurnoutRisk,,,
High,36,78,77
Low,28,0,73
Medium,258,98,822


## Save

In [5]:
df.to_csv('../data/processed/full_scored.csv', index=False)
print('Saved -> ../data/processed/full_scored.csv')

Saved -> ../data/processed/full_scored.csv
